# 01 — Project Initialization and PySpark Setup

This notebook is limited to the first phase of the project:

1. initialize the project environment;
2. start a local PySpark session;
3. define input/output paths;
4. load the raw citation edge list;
5. perform minimal cleaning;
6. save the cleaned edge list for the next notebooks.

No graph analysis, no centrality measures, and no plots are included here.

## 1. Imports

In [1]:
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, LongType

## 2. Project paths

Expected structure:

```text
project-root/
│
├── data/
│   ├── raw/
│   │   ├── cit-HepTh.txt
│   │   └── cit-HepTh-dates.txt
│   │
│   └── processed/
│
├── notebooks/
│   └── 01_project_initialization_pyspark.ipynb
│
└── requirements.txt
```

In [2]:

PROJECT_ROOT = Path("..").resolve()

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

NETWORK_FILE = RAW_DIR / "cit-HepTh.txt"
DATES_FILE = RAW_DIR / "cit-HepTh-dates.txt"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data directory: {RAW_DIR}")
print(f"Processed data directory: {PROCESSED_DIR}")
print(f"Network file exists: {NETWORK_FILE.exists()}")
print(f"Dates file exists: {DATES_FILE.exists()}")

Project root: /Users/chiarabelli/Desktop/citation-network-project
Raw data directory: /Users/chiarabelli/Desktop/citation-network-project/data/raw
Processed data directory: /Users/chiarabelli/Desktop/citation-network-project/data/processed
Network file exists: True
Dates file exists: True


## 3. Start Spark session

In [3]:
spark = (
    SparkSession.builder
    .appName("CitHepTh_Project_Initialization")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Spark session initialized.")
print(f"Spark version: {spark.version}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/15 11:23:36 WARN Utils: Your hostname, MacBook-Pro-di-Chiara.local, resolves to a loopback address: 127.0.0.1; using 10.208.162.29 instead (on interface en0)
26/05/15 11:23:36 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/15 11:23:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark session initialized.
Spark version: 4.1.1


## 4. Load raw citation network

The raw file contains comment lines starting with `#`.

Each valid row contains two paper identifiers:

- `from_node_id`: citing paper;
- `to_node_id`: cited paper.

In [4]:
raw_lines = spark.read.text(str(NETWORK_FILE))

raw_lines.show(5, truncate=False)

+-----------------------------------------------------------------------------+
|value                                                                        |
+-----------------------------------------------------------------------------+
|# Directed graph (each unordered pair of nodes is saved once): Cit-HepTh.txt |
|# Paper citation network of Arxiv High Energy Physics Theory category        |
|# Nodes: 27770 Edges: 352807                                                 |
|# FromNodeId\tToNodeId                                                       |
|1001\t9304045                                                                |
+-----------------------------------------------------------------------------+
only showing top 5 rows


## 5. Minimal cleaning

Cleaning steps performed in this notebook:

1. remove comment lines;
2. remove empty lines;
3. split each row into source and target node identifiers;
4. cast identifiers to integers;
5. remove duplicated edges;
6. remove self-loops, if any.

In [5]:
edge_schema = StructType([
    StructField("from_node_id", LongType(), nullable=False),
    StructField("to_node_id", LongType(), nullable=False),
])

edges = (
    raw_lines
    .filter(~F.col("value").startswith("#"))
    .filter(F.length(F.trim(F.col("value"))) > 0)
    .select(
        F.split(F.trim(F.col("value")), r"\s+").getItem(0).cast("long").alias("from_node_id"),
        F.split(F.trim(F.col("value")), r"\s+").getItem(1).cast("long").alias("to_node_id"),
    )
    .filter(F.col("from_node_id").isNotNull() & F.col("to_node_id").isNotNull())
    .dropDuplicates(["from_node_id", "to_node_id"])
    .filter(F.col("from_node_id") != F.col("to_node_id"))
)

edges.show(10)

+------------+----------+
|from_node_id|to_node_id|
+------------+----------+
|        1001|   9308122|
|        1001|   9505054|
|        1001|   9508094|
|        1001|   9603003|
|        1001|   9605222|
|        1001|   9607207|
|        1001|   9703082|
|        1001|   9802194|
|        1001|   9909120|
|     9311042|   9301043|
+------------+----------+
only showing top 10 rows


## 6. Basic loading summary

This is only a sanity check to verify that the dataset was loaded correctly.

More detailed graph statistics will be computed in the next notebooks.

In [6]:
summary_data = [
    ("Raw lines", raw_lines.count()),
    ("Cleaned edges", edges.count()),
    ("Distinct source nodes", edges.select("from_node_id").distinct().count()),
    ("Distinct target nodes", edges.select("to_node_id").distinct().count()),
    (
        "Distinct nodes",
        edges.select(F.col("from_node_id").alias("node_id"))
             .union(edges.select(F.col("to_node_id").alias("node_id")))
             .distinct()
             .count()
    ),
]

summary_df = spark.createDataFrame(summary_data, ["Metric", "Value"])

summary_df.show(truncate=False)

+---------------------+------+
|Metric               |Value |
+---------------------+------+
|Raw lines            |352811|
|Cleaned edges        |352768|
|Distinct source nodes|25055 |
|Distinct target nodes|23176 |
|Distinct nodes       |27769 |
+---------------------+------+



## 7. Save cleaned edge list

The cleaned network is saved in Parquet format.

This file will be used as input for the next notebooks.

In [7]:
CLEAN_EDGES_PATH = PROCESSED_DIR / "cit_hepth_clean_edges.parquet"

(
    edges
    .write
    .mode("overwrite")
    .parquet(str(CLEAN_EDGES_PATH))
)

print(f"Cleaned edge list saved to: {CLEAN_EDGES_PATH}")

Cleaned edge list saved to: /Users/chiarabelli/Desktop/citation-network-project/data/processed/cit_hepth_clean_edges.parquet


## 8. Optional: load paper dates without analysis

This section only checks whether the dates file can be loaded.

The temporal analysis will be performed in a separate notebook.

In [8]:
if DATES_FILE.exists():
    raw_dates = spark.read.text(str(DATES_FILE))

    dates = (
        raw_dates
        .filter(~F.col("value").startswith("#"))
        .filter(F.length(F.trim(F.col("value"))) > 0)
        .select(
            F.split(F.trim(F.col("value")), r"\s+").getItem(0).cast("long").alias("node_id"),
            F.split(F.trim(F.col("value")), r"\s+").getItem(1).alias("submission_date")
        )
        .filter(F.col("node_id").isNotNull() & F.col("submission_date").isNotNull())
        .dropDuplicates(["node_id"])
    )

    dates.show(10, truncate=False)

    CLEAN_DATES_PATH = PROCESSED_DIR / "cit_hepth_dates.parquet"
    dates.write.mode("overwrite").parquet(str(CLEAN_DATES_PATH))

    print(f"Dates saved to: {CLEAN_DATES_PATH}")
else:
    print("Dates file not found. Skipping this optional step.")

+-------+---------------+
|node_id|submission_date|
+-------+---------------+
|1001   |2000-01-01     |
|1002   |2000-01-01     |
|1003   |2000-01-01     |
|1004   |2000-01-02     |
|1005   |2000-01-03     |
|1006   |2000-01-02     |
|1007   |2000-01-02     |
|1008   |2000-01-02     |
|1009   |2000-01-02     |
|1010   |2000-01-02     |
+-------+---------------+
only showing top 10 rows
Dates saved to: /Users/chiarabelli/Desktop/citation-network-project/data/processed/cit_hepth_dates.parquet


## 9. Stop Spark session

Run this cell only when you have finished working with Spark in this notebook.

In [9]:
spark.stop()